# Μάθημα 09 - Πρότυπο Σχεδίασης Μεταγνωσίας


## Ρύθμιση

Αυτό το σημειωματάριο επιδεικνύει το πρότυπο σχεδίασης Μεταγνώσης χρησιμοποιώντας το Microsoft Agent Framework.

**Απαιτούμενα:**
- Ανάπτυξη Azure OpenAI διαμορφωμένη μέσω μεταβλητών περιβάλλοντος
- Azure CLI αυθεντικοποιημένο (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Τι είναι η μεταγνωσία;

Η μεταγνωσία είναι **σκέψη για τη σκέψη**. Στο πλαίσιο των πρακτόρων τεχνητής νοημοσύνης, σημαίνει την κατασκευή πρακτόρων που μπορούν:

- **Αναστοχάζονται** τα δικά τους αποτελέσματα και τη διαδικασία συλλογισμού
- **Εντοπίζουν σφάλματα** και ανακάμπτουν ομαλά αντί να αποτυγχάνουν σιωπηρά
- **Αξιολογούν** κατά πόσο οι απαντήσεις τους είναι πλήρεις και χρήσιμες
- **Προσαρμόζουν** τη στρατηγική τους όταν μια αρχική προσέγγιση δεν λειτουργεί (π.χ., επιστροφή σε εφεδρικό σύστημα)

Ένας μεταγνωστικός πράκτορας δεν απαντά απλώς σε ερωτήσεις — παρακολουθεί την απόδοσή του και προσαρμόζεται άμεσα.


## Κύρια και Εφεδρικά Εργαλεία

Ένα κοινό μοτίβο μεταγνωσίας είναι η **στρατηγική εφεδρείας**. Ο πράκτορας δοκιμάζει πρώτα ένα κύριο εργαλείο; αν αυτό αποτύχει (π.χ., σφάλμα 404), ο πράκτορας αναγνωρίζει την αποτυχία και διαφανώς μεταβαίνει σε ένα εφεδρικό εργαλείο.

Αυτό αντικατοπτρίζει συστήματα του πραγματικού κόσμου όπου οι κύριες υπηρεσίες ενδέχεται να μην είναι διαθέσιμες και οι πράκτορες πρέπει να αυτοδιαγνώσουν το ζήτημα πριν επιλέξουν μια εναλλακτική διαδρομή.

Παρακάτω ορίζουμε δύο εργαλεία αναζήτησης πτήσεων:
- **Κύριο** — καλύπτει το Παρίσι, το Τόκιο και τη Βαρκελώνη
- **Εφεδρικό** — καλύπτει το Βερολίνο, το Σίδνεϊ και τη Νέα Υόρκη


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Αυτο-αναστοχαστικός πράκτορας με ανάκτηση από σφάλματα

Ο πράκτορας παρακάτω έχει εντολή να δοκιμάσει πρώτα το κύριο σύστημα πτήσης, να αναγνωρίσει αποτυχίες και να μεταβεί διαφανώς στο εφεδρικό σύστημα. Μετά από κάθε απάντηση αξιολογεί σύντομα αν απάντησε πλήρως στην ερώτηση του χρήστη.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Πρότυπο Αυτοαξιολόγησης

Άλλη όψη της μεταγνώσης είναι η **αυτοαξιολόγηση**: ένας ξεχωριστός πράκτορας (ή ο ίδιος πράκτορας σε δεύτερο πέρασμα) εξετάζει μια απάντηση ως προς την πληρότητα, την ακρίβεια και τη χρησιμότητα.

Παρακάτω δημιουργούμε έναν πράκτορα `ResponseEvaluator` που βαθμολογεί τις απαντήσεις του ταξιδιωτικού πράκτορα σε τρεις διαστάσεις.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Περίληψη

Σε αυτό το μάθημα μάθατε πώς να δημιουργείτε **μεταγνωστικούς πράκτορες** χρησιμοποιώντας το Microsoft Agent Framework:

- **Αυτο-αναστοχασμός**: Πράκτορες που παρακολουθούν τη δική τους συλλογιστική και επικοινωνούν με διαφάνεια τι συνέβη.
- **Ανάκτηση σφαλμάτων με εφεδρικές λύσεις**: Ένα πρότυπο κύριου + εφεδρικού εργαλείου όπου ο πράκτορας εντοπίζει αποτυχίες (π.χ., σφάλματα 404) και δοκιμάζει αυτόματα μια εναλλακτική πηγή.
- **Αυτο-αξιολόγηση**: Ένας ξεχωριστός πράκτορας αξιολογητής που βαθμολογεί τις απαντήσεις για πληρότητα, ακρίβεια και χρησιμότητα.

Αυτά τα πρότυπα καθιστούν τους πράκτορες πιο ανθεκτικούς, διαφανείς και αξιόπιστους — καίριες ιδιότητες για αναπτύξεις σε παραγωγή.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Αποποίηση ευθυνών:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία αυτόματης μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Παρότι καταβάλλουμε προσπάθειες για ακρίβεια, παρακαλούμε λάβετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν σφάλματα ή ανακρίβειες. Το πρωτότυπο έγγραφο στην αρχική του γλώσσα πρέπει να θεωρείται ως η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική μετάφραση από άνθρωπο. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
